# Access Log Parser

Apache/Nginx Combined Log 형식의 `access.log`를 파싱해 DataFrame과 CSV로 저장한다.

이 노트북의 역할은 **파싱만** 담당한다.

- 원본 위치 추적: `line_no`, `raw_line`
- 원본/변환 시간 분리: `time_raw`, `time`
- 요청 형식 분류: `STANDARD_HTTP`, `EMPTY_REQUEST`, `NON_STANDARD_REQUEST`
- 파싱 결과 분류: `SUCCESS`, `PARTIAL`, `FAIL`
- 결과 저장: `data/interim/access_parsed.csv`
- 실패 행 저장: `data/output/access_parse_failures.csv`


In [2]:
# 라이브러리
import os
import re
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

## 1. 입력·출력 경로 설정

기본 프로젝트 구조에서는 `data/raw/access.log`를 자동으로 찾는다. 파일명이 `access_log.txt`인 경우도 지원한다.

다른 위치의 파일을 사용할 때는 실행 전에 환경변수 `ACCESS_LOG_PATH`를 지정할 수 있다.


In [4]:
def find_access_log():
    """프로젝트 구조와 현재 작업 위치를 고려해 access log 파일을 찾는다.
    환경변수 ACCESS_LOG_PATH가 설정되어 있으면 해당 경로를 우선 사용한다.
    """
    
    env_path = os.getenv("ACCESS_LOG_PATH")

    candidates = []
    
    if env_path:
        candidates.append(Path(env_path).expanduser())

    candidates.extend([
        Path("../data/raw/access.log"),
    ])

    for candidate in candidates:
        candidate = candidate.resolve()

        # 파일 찾으면 리턴
        if candidate.is_file():
            return candidate

    checked = "\n".join(f"- {path.resolve()}" for path in candidates)
    raise FileNotFoundError(
        "access log 파일을 찾지 못했습니다. 확인한 경로:\n" + checked
    )


access_file_path = find_access_log()

# 일반적인 data/raw 구조라면 data/interim, data/output을 사용한다.
# 첨부 파일처럼 별도 위치라면 해당 파일 옆에 interim, output 폴더를 만든다.
if access_file_path.parent.name == "raw":
    data_root = access_file_path.parent.parent
else:
    data_root = access_file_path.parent

interim_dir = Path(os.getenv("ACCESS_INTERIM_DIR", data_root / "interim")).resolve()
output_dir = Path(os.getenv("ACCESS_OUTPUT_DIR", data_root / "output")).resolve()

interim_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

parsed_csv_path = interim_dir / "access_parsed.csv"
failure_csv_path = output_dir / "access_parse_failures.csv"

print(f"입력 파일: {access_file_path}")
print(f"파싱 결과: {parsed_csv_path}")
print(f"실패 결과: {failure_csv_path}")

입력 파일: C:\Users\seoyeon\log-dlq-pipeline\data\raw\access.log
파싱 결과: C:\Users\seoyeon\log-dlq-pipeline\data\interim\access_parsed.csv
실패 결과: C:\Users\seoyeon\log-dlq-pipeline\data\output\access_parse_failures.csv


## 2. 로그 정규식

Apache/Nginx Combined Log의 다음 필드를 추출한다.

`ip ident authuser [time] "request" status size "referer" "agent"`


In [6]:
# Acess Log 정규식

ACCESS_PATTERN = re.compile(
    r'^\s*'
    r'(?P<ip>\S+)\s+'
    r'(?P<ident>\S+)\s+'
    r'(?P<authuser>\S+)\s+'
    r'\[(?P<time_raw>[^\]]+)\]\s+'
    r'"(?P<request>(?:\\.|[^"\\])*)"\s+'
    r'(?P<status>\d{3}|-)\s+'
    r'(?P<size>\d+|-)\s+'
    r'"(?P<referer>(?:\\.|[^"\\])*)"\s+'
    r'"(?P<agent>(?:\\.|[^"\\])*)"'
    r'\s*$'
)

HTTP_METHOD_PATTERN = re.compile(r'^[A-Z][A-Z0-9_-]*$')
HTTP_PROTOCOL_PATTERN = re.compile(r'^HTTP/\d+(?:\.\d+)?$')

## 3. 필드 변환 함수

`request`는 원문을 그대로 보존하면서 `method`, `path`, `protocol`로 분리한다.


In [8]:
# 최종 DataFrame 컬럼
ACCESS_COLUMNS = [
    "line_no",
    "raw_line",

    "ip",
    "ident",
    "authuser",

    "time_raw",
    "time",

    "request",
    "method",
    "path",
    "protocol",

    "status",
    "size",
    "referer",
    "agent",

    "parse_status",
    "request_type",
    "parse_note",
]

In [9]:
# 공통 변환 함수

def normalize_placeholder(value):
    """
    Combined Log의 '-' 값을 결측치 None으로 변환한다.
    원본 값은 raw_line에 그대로 보존된다.
    """

    if value == "-":
        return None

    return value


def parse_timestamp(value):
    """
    Apache timestamp를 pandas Timestamp로 변환한다.

    변환 실패 시 NaT를 반환한다.
    """

    return pd.to_datetime(
        value,
        format="%d/%b/%Y:%H:%M:%S %z",
        errors="coerce",
    )

In [10]:
# Requset 파싱 함수
def parse_request(request):
    """
    request를 method/path/protocol로 나누고 형식을 분류한다.

    request_type:
    - STANDARD_HTTP
    - EMPTY_REQUEST
    - NON_STANDARD_REQUEST

    비표준 요청은 method 통계에 섞이지 않도록
    method/path/protocol을 None으로 반환한다.
    """

    if request is None:
        return {
            "method": None,
            "path": None,
            "protocol": None,
            "request_type": "EMPTY_REQUEST",
        }

    request = request.strip()

    if request in {"", "-"}:
        return {
            "method": None,
            "path": None,
            "protocol": None,
            "request_type": "EMPTY_REQUEST",
        }

    parts = request.split()

    if len(parts) != 3:
        return {
            "method": None,
            "path": None,
            "protocol": None,
            "request_type": "NON_STANDARD_REQUEST",
        }

    method, path, protocol = parts

    is_standard_method = bool(
        HTTP_METHOD_PATTERN.fullmatch(method)
    )

    is_standard_protocol = bool(
        HTTP_PROTOCOL_PATTERN.fullmatch(protocol)
    )

    is_standard_path = bool(path)

    if not (
        is_standard_method
        and is_standard_path
        and is_standard_protocol
    ):
        return {
            "method": None,
            "path": None,
            "protocol": None,
            "request_type": "NON_STANDARD_REQUEST",
        }

    return {
        "method": method,
        "path": path,
        "protocol": protocol,
        "request_type": "STANDARD_HTTP",
    }

In [11]:
# 빈 레코드 생성 함수

def create_empty_record(line_no, raw_line):
    """
    모든 컬럼이 포함된 기본 레코드를 생성한다.

    파싱에 실패하더라도 DataFrame 컬럼 구조가 달라지지 않도록 한다.
    """

    record = {
        column: None
        for column in ACCESS_COLUMNS
    }

    record["line_no"] = line_no
    record["raw_line"] = raw_line

    return record

## 4. 한 줄 파싱 함수

매칭 실패 행도 버리지 않고 `FAIL` 레코드로 남긴다. 그래야 나중에 원본 행을 바로 추적할 수 있다.


In [13]:
# Access Log 한 줄 파싱

def parse_access_line(line, line_no):
    """
    access log 한 줄을 구조화된 dict로 변환한다.

    parse_status:
    - SUCCESS: 기본 로그 구조와 필드 변환 성공
    - PARTIAL: 기본 구조는 읽었으나 필수 필드 일부 변환 실패
    - FAIL: Combined Log 형식 자체를 읽지 못함

    request_type은 parse_status와 별도로 관리한다.
    """

    raw_line = line.rstrip("\r\n")

    record = create_empty_record(
        line_no=line_no,
        raw_line=raw_line,
    )

    # 빈 줄
    if raw_line == "":
        record["parse_status"] = "FAIL"
        record["parse_note"] = "EMPTY_LINE"

        return record

    match = ACCESS_PATTERN.fullmatch(raw_line)

    # Combined Log 구조 불일치
    if match is None:
        record["parse_status"] = "FAIL"
        record["parse_note"] = "COMBINED_LOG_PATTERN_MISMATCH"

        return record

    data = match.groupdict()

    parsed_time = parse_timestamp(
        data["time_raw"]
    )

    request_fields = parse_request(
        data["request"]
    )

    notes = []

    # timestamp는 필수 분석 필드이므로 실패 시 PARTIAL
    if pd.isna(parsed_time):
        notes.append("INVALID_TIMESTAMP")

    # status가 '-'라면 구조는 읽었지만 필수값이 없으므로 PARTIAL
    if data["status"] == "-":
        parsed_status = None
        notes.append("MISSING_STATUS")
    else:
        parsed_status = int(data["status"])

    # size의 '-'는 정상적으로 발생할 수 있으므로 PARTIAL로 처리하지 않음
    if data["size"] == "-":
        parsed_size = None
    else:
        parsed_size = int(data["size"])

    parse_status = (
        "PARTIAL"
        if notes
        else "SUCCESS"
    )

    record.update({
        "ip": data["ip"],

        "ident": normalize_placeholder(
            data["ident"]
        ),

        "authuser": normalize_placeholder(
            data["authuser"]
        ),

        "time_raw": data["time_raw"],
        "time": parsed_time,

        # request 원문은 '-'도 그대로 보존
        "request": data["request"],

        "method": request_fields["method"],
        "path": request_fields["path"],
        "protocol": request_fields["protocol"],

        "status": parsed_status,
        "size": parsed_size,

        "referer": normalize_placeholder(
            data["referer"]
        ),

        "agent": normalize_placeholder(
            data["agent"]
        ),

        "parse_status": parse_status,
        "request_type": request_fields["request_type"],

        "parse_note": (
            ";".join(notes)
            if notes
            else None
        ),
    })

    return record

## 5. 전체 파일 파싱


In [15]:
# Access Log 전체 파일 파싱

def parse_access_file(file_path):
    """
    access log 전체를 읽어 DataFrame으로 반환한다.
    """

    file_path = Path(file_path)

    records = []

    with file_path.open(
        "r",
        encoding="utf-8",
        errors="replace",
    ) as file:

        for line_no, line in enumerate(
            file,
            start=1,
        ):
            record = parse_access_line(
                line=line,
                line_no=line_no,
            )

            missing_keys = (
                set(ACCESS_COLUMNS)
                - set(record.keys())
            )

            unexpected_keys = (
                set(record.keys())
                - set(ACCESS_COLUMNS)
            )

            assert not missing_keys, (
                f"{line_no}번째 레코드에 누락된 키가 있습니다: "
                f"{missing_keys}"
            )

            assert not unexpected_keys, (
                f"{line_no}번째 레코드에 예상하지 않은 키가 있습니다: "
                f"{unexpected_keys}"
            )

            records.append(record)

    frame = pd.DataFrame.from_records(
        records,
        columns=ACCESS_COLUMNS,
    )

    frame["line_no"] = pd.to_numeric(
        frame["line_no"],
        errors="raise",
    ).astype("Int64")

    frame["status"] = pd.to_numeric(
        frame["status"],
        errors="coerce",
    ).astype("Int64")

    frame["size"] = pd.to_numeric(
        frame["size"],
        errors="coerce",
    ).astype("Int64")

    return frame

In [16]:
# 전체 파일 실행

df = parse_access_file(
    access_file_path
)

print(f"전체 로그 수: {len(df):,}")

display(
    df.head()
)

전체 로그 수: 5,143


,line_no,raw_line,ip,ident,authuser,time_raw,time,request,method,path,protocol,status,size,referer,agent,parse_status,request_type,parse_note
0,1,"210.110.68.35 - - [12/Jun/2025:13:35:37 +0900] ""GET / HTTP/1.1"" 403 7620 ""-"" ""Mozilla/5.0 (Windows NT 10.0; Win64; x...",210.110.68.35,None,None,12/Jun/2025:13:35:37 +0900,2025-06-12 13:35:37+09:00,GET / HTTP/1.1,GET,/,HTTP/1.1,403,7620,None,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36",SUCCESS,STANDARD_HTTP,None
1,2,"210.110.68.35 - - [12/Jun/2025:13:35:37 +0900] ""GET /icons/poweredby.png HTTP/1.1"" 200 15443 ""http://210.110.104.16/...",210.110.68.35,None,None,12/Jun/2025:13:35:37 +0900,2025-06-12 13:35:37+09:00,GET /icons/poweredby.png HTTP/1.1,GET,/icons/poweredby.png,HTTP/1.1,200,15443,http://210.110.104.16/,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36",SUCCESS,STANDARD_HTTP,None
2,3,"210.110.68.35 - - [12/Jun/2025:13:35:37 +0900] ""GET /poweredby.png HTTP/1.1"" 200 5714 ""http://210.110.104.16/"" ""Mozi...",210.110.68.35,None,None,12/Jun/2025:13:35:37 +0900,2025-06-12 13:35:37+09:00,GET /poweredby.png HTTP/1.1,GET,/poweredby.png,HTTP/1.1,200,5714,http://210.110.104.16/,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36",SUCCESS,STANDARD_HTTP,None
3,4,"210.110.68.35 - - [12/Jun/2025:13:35:37 +0900] ""GET /favicon.ico HTTP/1.1"" 404 196 ""http://210.110.104.16/"" ""Mozilla...",210.110.68.35,None,None,12/Jun/2025:13:35:37 +0900,2025-06-12 13:35:37+09:00,GET /favicon.ico HTTP/1.1,GET,/favicon.ico,HTTP/1.1,404,196,http://210.110.104.16/,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36",SUCCESS,STANDARD_HTTP,None
4,5,"210.110.68.35 - - [12/Jun/2025:13:36:26 +0900] ""-"" 408 - ""-"" ""-""",210.110.68.35,None,None,12/Jun/2025:13:36:26 +0900,2025-06-12 13:36:26+09:00,-,None,None,None,408,<NA>,None,None,SUCCESS,EMPTY_REQUEST,None


## 6. Parser 결과 확인

`EMPTY_REQUEST`는 로그 전체 형식이 깨진 것은 아니므로 `FAIL`이 아니라 `PARTIAL`로 둔다. 데이터 품질상 정상인지 여부는 2주차 규칙 설계에서 결정한다.


In [18]:
# Parser 상태 요약

def make_distribution(
    frame,
    column,
):
    """
    지정한 컬럼의 건수와 비율을 계산한다.
    """

    result = (
        frame[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="count")
    )

    if len(frame) == 0:
        result["rate_pct"] = 0.0
    else:
        result["rate_pct"] = (
            result["count"]
            / len(frame)
            * 100
        ).round(2)

    return result


parse_summary = make_distribution(
    df,
    "parse_status",
)

request_summary = make_distribution(
    df,
    "request_type",
)


print("[Parse Status]")

display(
    parse_summary
)


print("[Request Type]")

display(
    request_summary
)

[Parse Status]


,parse_status,count,rate_pct
0,SUCCESS,5143,100.0


[Request Type]


,request_type,count,rate_pct
0,STANDARD_HTTP,4579,89.03
1,NON_STANDARD_REQUEST,519,10.09
2,EMPTY_REQUEST,45,0.87


In [19]:
# 유형별 예시 확인

sample_columns = [
    "line_no",
    "status",
    "request_type",
    "parse_status",
    "parse_note",
    "request",
    "raw_line",
]


print("[EMPTY_REQUEST 예시]")

empty_request_rows = df.loc[
    df["request_type"].eq("EMPTY_REQUEST"),
    sample_columns,
]

print(f"전체 건수: {len(empty_request_rows):,}")

display(
    empty_request_rows.head(10)
)


print("[NON_STANDARD_REQUEST 예시]")

non_standard_rows = df.loc[
    df["request_type"].eq("NON_STANDARD_REQUEST"),
    sample_columns,
]

print(f"전체 건수: {len(non_standard_rows):,}")

display(
    non_standard_rows.head(10)
)


print("[PARTIAL 예시]")

partial_rows = df.loc[
    df["parse_status"].eq("PARTIAL"),
    sample_columns,
]

print(f"전체 건수: {len(partial_rows):,}")

display(
    partial_rows.head(10)
)


print("[FAIL 예시]")

fail_rows = df.loc[
    df["parse_status"].eq("FAIL"),
    sample_columns,
]

print(f"전체 건수: {len(fail_rows):,}")

display(
    fail_rows.head(10)
)

[EMPTY_REQUEST 예시]
전체 건수: 45


,line_no,status,request_type,parse_status,parse_note,request,raw_line
4,5,408,EMPTY_REQUEST,SUCCESS,None,-,"210.110.68.35 - - [12/Jun/2025:13:36:26 +0900] ""-"" 408 - ""-"" ""-"""
10,11,408,EMPTY_REQUEST,SUCCESS,None,-,"210.110.68.35 - - [12/Jun/2025:16:46:29 +0900] ""-"" 408 - ""-"" ""-"""
12,13,408,EMPTY_REQUEST,SUCCESS,None,-,"210.110.68.35 - - [12/Jun/2025:16:49:34 +0900] ""-"" 408 - ""-"" ""-"""
17,18,408,EMPTY_REQUEST,SUCCESS,None,-,"210.110.64.41 - - [12/Jun/2025:16:58:03 +0900] ""-"" 408 - ""-"" ""-"""
18,19,408,EMPTY_REQUEST,SUCCESS,None,-,"210.110.64.41 - - [12/Jun/2025:16:58:03 +0900] ""-"" 408 - ""-"" ""-"""
31,32,408,EMPTY_REQUEST,SUCCESS,None,-,"220.67.241.146 - - [12/Jun/2025:17:18:36 +0900] ""-"" 408 - ""-"" ""-"""
38,39,408,EMPTY_REQUEST,SUCCESS,None,-,"210.110.68.35 - - [12/Jun/2025:17:41:57 +0900] ""-"" 408 - ""-"" ""-"""
60,61,408,EMPTY_REQUEST,SUCCESS,None,-,"220.67.241.146 - - [12/Jun/2025:17:43:44 +0900] ""-"" 408 - ""-"" ""-"""
61,62,408,EMPTY_REQUEST,SUCCESS,None,-,"210.110.64.41 - - [12/Jun/2025:17:43:55 +0900] ""-"" 408 - ""-"" ""-"""
62,63,408,EMPTY_REQUEST,SUCCESS,None,-,"210.110.68.35 - - [12/Jun/2025:17:44:16 +0900] ""-"" 408 - ""-"" ""-"""


[NON_STANDARD_REQUEST 예시]
전체 건수: 519


,line_no,status,request_type,parse_status,parse_note,request,raw_line
285,286,400,NON_STANDARD_REQUEST,SUCCESS,None,SSH-2.0-WanScannerBot,"91.90.126.37 - - [16/Jun/2025:17:46:20 +0900] ""SSH-2.0-WanScannerBot"" 400 226 ""-"" ""-"""
286,287,400,NON_STANDARD_REQUEST,SUCCESS,None,\x03,"91.90.126.37 - - [16/Jun/2025:17:46:20 +0900] ""\x03"" 400 226 ""-"" ""-"""
287,288,400,NON_STANDARD_REQUEST,SUCCESS,None,\x10\x0f,"91.90.126.37 - - [16/Jun/2025:17:46:21 +0900] ""\x10\x0f"" 400 226 ""-"" ""-"""
288,289,400,NON_STANDARD_REQUEST,SUCCESS,None,OPTIONS / RTSP/1.0,"91.90.126.37 - - [16/Jun/2025:17:46:30 +0900] ""OPTIONS / RTSP/1.0"" 400 226 ""-"" ""-"""
289,290,400,NON_STANDARD_REQUEST,SUCCESS,None,\x01,"91.90.126.37 - - [16/Jun/2025:17:46:35 +0900] ""\x01"" 400 226 ""-"" ""-"""
291,292,400,NON_STANDARD_REQUEST,SUCCESS,None,\x16\x03\x01,"34.94.93.67 - - [16/Jun/2025:19:01:39 +0900] ""\x16\x03\x01"" 400 226 ""-"" ""-"""
307,308,400,NON_STANDARD_REQUEST,SUCCESS,None,\x16\x03\x01,"159.223.171.113 - - [17/Jun/2025:01:52:54 +0900] ""\x16\x03\x01"" 400 226 ""-"" ""-"""
308,309,400,NON_STANDARD_REQUEST,SUCCESS,None,\x16\x03\x01,"159.223.171.113 - - [17/Jun/2025:01:52:55 +0900] ""\x16\x03\x01"" 400 226 ""-"" ""-"""
309,310,400,NON_STANDARD_REQUEST,SUCCESS,None,\x16\x03\x01,"159.223.171.113 - - [17/Jun/2025:01:52:55 +0900] ""\x16\x03\x01"" 400 226 ""-"" ""-"""
341,342,400,NON_STANDARD_REQUEST,SUCCESS,None,OPTIONS / RTSP/1.0,"172.236.137.159 - - [17/Jun/2025:09:04:06 +0900] ""OPTIONS / RTSP/1.0"" 400 226 ""-"" ""-"""


[PARTIAL 예시]
전체 건수: 0


,line_no,status,request_type,parse_status,parse_note,request,raw_line


[FAIL 예시]
전체 건수: 0


,line_no,status,request_type,parse_status,parse_note,request,raw_line


## 7. CSV 저장

- 전체 파싱 결과: `access_parsed.csv`
- 정규식 매칭 실패 행: `access_parse_failures.csv`

`PARTIAL` 행은 전체 결과에 보존한다. 1주차에는 데이터 오류로 확정하지 않는다.


In [21]:
# csv 저장
df.to_csv(
    parsed_csv_path,
    index=False,
    encoding="utf-8-sig",
)


failure_columns = [
    "line_no",
    "raw_line",
    "parse_status",
    "parse_note",
]


failures = df.loc[
    df["parse_status"].eq("FAIL"),
    failure_columns,
].copy()


failures.to_csv(
    failure_csv_path,
    index=False,
    encoding="utf-8-sig",
)


print(
    f"저장 완료: {parsed_csv_path} "
    f"({len(df):,}행)"
)

print(
    f"저장 완료: {failure_csv_path} "
    f"({len(failures):,}행)"
)

저장 완료: C:\Users\seoyeon\log-dlq-pipeline\data\interim\access_parsed.csv (5,143행)
저장 완료: C:\Users\seoyeon\log-dlq-pipeline\data\output\access_parse_failures.csv (0행)


## 8. 기본 검증

원본 행 수 보존, 행 번호 중복 여부, 필수 컬럼 존재 여부를 확인한다.


In [23]:
with access_file_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
) as file:
    raw_line_count = sum(
        1
        for _ in file
    )


# 원본의 모든 행이 DataFrame에 들어왔는지 확인
assert len(df) == raw_line_count, (
    "원본 행 수와 DataFrame 행 수가 다릅니다."
)


# 컬럼이 정확하게 일치하는지 확인
assert df.columns.tolist() == ACCESS_COLUMNS, (
    "DataFrame 컬럼 또는 컬럼 순서가 ACCESS_COLUMNS와 다릅니다."
)


# 원본 위치 추적용 line_no 확인
assert df["line_no"].is_unique, (
    "line_no가 중복되었습니다."
)

assert df["line_no"].tolist() == list(
    range(1, len(df) + 1)
), (
    "line_no가 원본 행 순서와 일치하지 않습니다."
)


# 원본 문자열 보존 확인
assert df["raw_line"].notna().all(), (
    "raw_line에 결측치가 있습니다."
)


# Parser 상태값 확인
allowed_parse_status = {
    "SUCCESS",
    "PARTIAL",
    "FAIL",
}

assert df["parse_status"].isin(
    allowed_parse_status
).all(), (
    "허용되지 않은 parse_status가 있습니다."
)


# Request 유형 확인
allowed_request_types = {
    "STANDARD_HTTP",
    "EMPTY_REQUEST",
    "NON_STANDARD_REQUEST",
}

parsed_request_types = set(
    df.loc[
        df["parse_status"].ne("FAIL"),
        "request_type",
    ].dropna()
)

assert parsed_request_types.issubset(
    allowed_request_types
), (
    "허용되지 않은 request_type이 있습니다."
)


# FAIL 건수와 실패 CSV 건수 비교
assert len(failures) == int(
    df["parse_status"].eq("FAIL").sum()
), (
    "FAIL 건수와 실패 CSV 건수가 다릅니다."
)


# STANDARD_HTTP만 method/path/protocol을 가져야 함
non_standard_has_method = df.loc[
    df["request_type"].ne("STANDARD_HTTP"),
    ["method", "path", "protocol"],
].notna().any(axis=1)

assert not non_standard_has_method.any(), (
    "비표준 요청의 method/path/protocol에 값이 들어 있습니다."
)


print("기본 검증 통과")

print(
    f"- 원본/DF 행 수: {raw_line_count:,}"
)

print(
    f"- line_no 고유성: {df['line_no'].is_unique}"
)

print(
    f"- 최종 컬럼 수: {len(df.columns)}"
)

기본 검증 통과
- 원본/DF 행 수: 5,143
- line_no 고유성: True
- 최종 컬럼 수: 18


## 최종 DataFrame 컬럼


In [25]:
df.info()
df.columns.tolist()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5143 entries, 0 to 5142
Data columns (total 18 columns):
 #   Column        Non-Null Count  Dtype                    
---  ------        --------------  -----                    
 0   line_no       5143 non-null   Int64                    
 1   raw_line      5143 non-null   object                   
 2   ip            5143 non-null   object                   
 3   ident         0 non-null      object                   
 4   authuser      0 non-null      object                   
 5   time_raw      5143 non-null   object                   
 6   time          5143 non-null   datetime64[ns, UTC+09:00]
 7   request       5143 non-null   object                   
 8   method        4579 non-null   object                   
 9   path          4579 non-null   object                   
 10  protocol      4579 non-null   object                   
 11  status        5143 non-null   Int64                    
 12  size          4876 non-null   Int6

['line_no',
 'raw_line',
 'ip',
 'ident',
 'authuser',
 'time_raw',
 'time',
 'request',
 'method',
 'path',
 'protocol',
 'status',
 'size',
 'referer',
 'agent',
 'parse_status',
 'request_type',
 'parse_note']